An LLM Eval method is the mechanish we use to decide wether an LLM's Output is good or not - The procedure that takes the output and produces a judgement about it

Types of LLM Evals:

    1 - Programmtic / Deterministic : 
        - The code produces the verdict
        - The output is checked against a rule or a known correct answer by a program
        - Exact match , regex , JSON Schema Validation or running the output and comparing the results

    2 - Human Based :
        - People rate , rank or compare the outputs directly
        - The most expensive and slowest but gold standard for nuanced , subjective or high staked judgements
    
    3 - Model Graded (LLM as a Judge) :
        - A LLM produced the verdict
        - A strong LLM scores or rank output against a rubric
        - Used when theres no single correct answer but we can describe what a "good/better" output looks like

[For eg We are evaluating a RAG Chatbot and in the specifically Retriever Component]

Programatic Evaluation Pipeline :

    1 - Define Task and Target { Task : Retriever working correctly for retrieveing most relevant documents , Target : Retriever component }
    2 - Define a success Criteria { Recall @ k = Out of all the correct/relevant items that exist , how many did the system retrieve in the "top k" Results}
    3 - Build a Dataset {Build a golden dataset which has questions and correct answers(Most relevant doc-ids)}
    4 - Define an Evaluation method
    5 - Run the model
    6 - Evaluate the results
    7 - Analyze the results

[Better Embedding model , Reranking , Query Expansion , Increase the value of k]

Types of Human Evals :

    1 - Red teaming - actively attacking the system to find failures: crafting jailbreaks, prompt injections, adversarial and edge-case inputs, trying to make it produce harmful or wrong output. Creative adversarial work automated evals can't originate.

    2 - A/B testing - real users as graders in production; their behavior (thumbs-up, task completion, re-asking, escalation) is the verdict on which version is better.

    3 - Direct grading/rating - a person reads outputs and scores them against a rubric (the human-eval pipeline you just built). The most basic role: human as the grader.

    4 - Gold-answer/dataset creator - writing or verifying the correct answers (gold SQL, gold doc IDs, concept rubrics) and curating the test set to cover the real distribution.
    
    5 - HITL (human-in-the-loop) - a human placed inside the production flow as a checkpoint, reviewing/approving/editing/rejecting live outputs before they reach the user - triggered on the risky slice (low-confidence, high-stakes, or flagged outputs).


    [MAE - Mean Absolute Error - for comparing 2 columns]


    a - Reference-based Evals :
    You have a reference - a known-correct answer (or the key things a correct answer must contain) written down in advance for each test case. You grade by comparing the output against that reference.

    b - Reference-free Evals :
    You have no predefined correct answer. You judge the output's quality directly, on its own terms, against a criterion or rubric - but a rubric here is a scale/standard ("what does a good answer look like"), not a per-item correct answer.

Why do we need Multiple Evaluation Pipelines

    - Multiple Failure points
        - Individual Component Level
        - Different Workflows
        - Or entire applications
    
    - Risk Categories
        - Quality
        - Safety
        - Operations (latency and other metrics)

Offline vs Online Evals :

Offline evals are the evals that is done on a ai-system during pre-deployment (before deploying the system)
It is used for measuring the correctness of the system and its components

Online evals are the evals that are done on a ai_system during post-deployment (after the system is deployed)
It is used for checking wether the application is running fine

Stages in Online Evals:

1 - Logging
- Captures a structured, replayable record of every conversation turn.
- Identity/threading: conversation_id, turn_id, user_id/session_id, timestamp.
- Input: raw student message, plus any preprocessing (normalized text, detected language/intent).

Retrieved context
- Output: response text, model_name/prompt_version (needed for A/B testing and "which version regressed"), tool calls, finish reason.
- Operational telemetry → this is the operational-metric family, captured directly here: latency_ms (ideally split into retrieval vs generation), prompt_tokens, completion_tokens, derived cost, error/status codes.
- Downstream user signals → this is the behavioral-metric family, captured directly here: thumbs up/down, escalation to support, drop-off, rephrase, conversion.

Engineering properties:
- Non-blocking - logging must never add latency to the response; fire to a queue, write in the background.
- Durable + queryable -  a warehouse or observability tool (for eg : LangSmith) you can run analytical queries against, not scattered text logs.
- Late-signal attachment - signals land at different times (thumbs-down in seconds, escalation in an hour, conversion the next day), so records are keyed on conversation_id and updated as signals arrive.
- PII handling - scrub/tokenize emails, phone numbers, payment details; apply retention limits and access control.

Metrices/Observations that need to be Computed (Produced by an evaluator) :

- Logging → Stratified Sampling → Evaluator → Dashboard → Alerting 

- Faithfulness
- Answer relevance
- Correctness
- Hallucination
- Toxicity
- Bias & fairness
- PII (Personal Information Identification) leakage/prompt injection
- Conciseness
- Task completion, satisfaction

Metrices/Observations Captured directly from trace (during logging or other steps) :

- Logging → Dashboard → Alerting

- Thumbs up/down
- Escalation rate
- Abandonment/drop-off
- Rephrase rate
- Conversion
- Latency (p50/p95/p99)
- Cost per conversation
- Token usage
- Error rate

- Multi-Benchmark Leaderboards - livebench.ai
- Multi-Benchmark Leaderboards [Cost per token/query , Response Latency , Context window size , Price-to-Performance ratio] - artificialanalysis.ai
- Human Prefernce Based Leaderboards - Ranks based on human votes (2 responsed from diff models and user select which response is better) - arena.ai
- llm-stats.com
- openrouter - for testing all the llms (opensource,openai,anthropic all models available)

Prompt Caching - Suppose  majority of time while running multiple query some portion of system prompt is always the same that goes to the LLM
(for eg : If we are a building a text-2-SQL AI Application - the system prompt always contains the DB Schema that is required by the LLM to generate SQL Queries corresponding to user input) - KV Caching

Task
Cost - Input tokens + output tokens 
Latency - 2-3 seconds
Context Window 
On prem models / service providers
correctness

Selct top 5-10 Best LLM Candidates


### Custom Model Evals - Evaluating which model is best for our text-2-Sql 

#### Steps:
- Create a Dataset :
    - Using Kaggle Dataset of Ipl matches from 2008-2024
- Filter and Load the Database :
    - Sqlite as Database service and filtering the data from 2020-2024 just for easiness
- Extract the Schemas of Tables :
    - Used for passing the [schema + user_query] in the System Prompt so that LLM Can generate a valid sql query
- Build the Golden Dataset :
    - Generating a set of 20-50 Questions along with the Proper SQL Queries of that questions so while testing different models we can compare the  SQL Queries generated by the specific LLM with the Queries present in Golden Dataset  
- Validate the golden dataset :
    - Run the Queries present in the golden dataset on the Database and validate wether the Queries generated are correct
- Store the golden dataset in CSV Format
- Just cross check the overall flow
- Pick Candidate models :
    - As discussed based on the following factors - { Task , Cost - Input tokens + output tokens , Latency - 2-3 seconds , Context Window , On prem models / service providers , correctness } choose the top 5-10 candidate LLM Models - { use llm-stats.com , as it ranks model based on the code generation cpabilities (we are treating generating sql queries as equivalent to writing code only) as there are no good sql-2-text benchmarking of llm's done }
- Build the Evaluator
- Build the orchestrator
- Run all the models
- Choose the final Model

### "How Do You Evaluate Your RAG App?" - Building a basic RAG Chatbot and Evaluating it at various levels

#### Three Main Levels of evaluation

##### 1 - Component level : Each component evaluated in isolation
- A - Retriever Component :

    - The basic work of retriever is to retrieve relevant documents based on the input query from the vector database

    - Given a question wether retriever is able to retrieve relevant documents from vector db?

    - It will be **Reference-based** Evaluation process , as there will be **set of questions and corresponding correct answers already present (golden dataset)** for evaluation ans we will use the **Programatic approach**

    - Evaluation matrices used :
        - Recall (Recall @ k) - How many documents our retriever was able to retrieve out of all correct documents that are related to the input query , for eg - there are 5 correct docs and retriever was able to retrieve 3 correct docs , so ***recall = 3 / 5***

        - Precision (Precision @ k) - Out of all the documents retrieved , how many of them were actually useful , for eg - Retriever retrieved total 5 documents and only 2 of them were relevant to the input query (Derived from the golden dataset) , so ***precision = 2 /5***

        - Note - Recall and precision have a trade-off , so if you tend to improve one metric (move closer towards 1) the other will decrease (goes towards 0) as if we try to improve the **Recall** , so we **increase the value of k** (retrieve top k documents from the vectore-store based on the similarity score of the input query and the vectors(Chunks) present in the vectore-store) the **Recall increases** but as value of k is increased some **documents which are not relevant to query are also fetched** which leads to **decrease in the value of Precision** , as amount of noisy documents increases

- B - Generator Component :
    - The work of generator is to generate relevant answers given the context (retrieved documents from retriever) and the user query

    - It will be a **Refernce-free** Evaluation processs , as we will use **LLM As a Judge** for the evaluation and it does not have golden dataset

    - **Evaluation matrices used :**

        - ***Faithfulness*** - It evaluates wether the answer generated is from the given context itself or LLM has hallucinated the answer

        - ***Answer Relevancy*** - It evaluater wether the generated answer is related to Input question or it drifts away from the points expected to be answered from the Input question

            - [ Input_ques + Context ] → GENERATOR → Answer
            - [ Input_ques + Answer ] → LLM as a Judge → Breaks down the Answer into CLAIMS → Then we prompt LLM to Check wether the CLAIMS are relevant to the Input_ques
            - Faithfulness score = no of relevant claims / total claims

        - ***Contextual Relevancy*** - It evaluates wether the retrieved context from the Retriever relevant to the Input Question asked

            - Input_ques → [ RETRIEVER ] → Generated Context → [ LLM as a Judge ] → Breaks the context into CLAIMS → Checks wether the claims are Relevant to the Input_ques Asked 
            - For eg. LLM generated 15 claims from context , and out of them 10 were related to the Input query , so contextual_relevancy = 10/15
    
- C - Whats the issue with the above evaluation methods?
    - As both of the evaluation metrics require Golden dataset which contains query and corresponding correct doc-ids of chunks where the answer belong to

    - Generating the golden dataset is very hectic process as there are numerous amount of chunks and to find correct chunk-ids requires a lot of human effort and time

    - And most importantly the chunks depends on the chunk_size and the chunk_overlap parameters

    - So if the chunk_size / chunk_overlap / or the data ( new data is added / deleted) the chunks changes and so the ids , so we have to recreate the golden dataset the previous one becomes vague and of no use

    - So not a feasible method for evaluation

- D - So what's the Proper Methods?
    
    - 1 : ContextualRecallMetric in DeepEval :
        
        - The golden dataset would contain the **questions** and the **Ideal Answers** [not fetched from google , written on the basis of Data] and not the chunk_ids representing the correct answers

        - In the next step the question would be passed to the **Retriever** and it will **retrieve top-k relevant documents** with **corresponding doc-ids**

        - Then the **ideal_answer present in thr golden ds** would be passed to **LLM** and ***LLM will break down the ideal answers into claims***

        - Then the **fetched relevant documents from retriever** would be **passed to the same LLM** and it is instructed to **find which documents contains the claims present in the ideal answers** , if **all claims are present** in the retrieved context then the ***Recall would be 1***

    - 2 : ContextualPrecisionMetric in DeepEval :

        - In this the LLM is prompted in the following ways - " Here is the question : {input_query}. Here is the ideal(Expected) answer : {ideal_answer} . Here is one Retrieved chunk from the retriever : {chunk_retrieved[i]}. Check wether the following content of chunk contain information that heps to produce the ideal(Expected) answer ? Answer Yes or No and give the reason for the same ".
    
        - It also considers the rank of the retrieved chunks

        - for eg : [23,45,67,89,34] are the retrieved chunk_ids and 45 , 89 are useful chunks which helps to produce the ideal_answer and rest do not
        
        - before Reranking -  [23(n) , 45(y), 67(n) , 89(y) , 34(n)]
            - 23(n) - precision (how many useful out of total docs fetched) =  0/1 = 0
            - 45(y) - 1/2 = 0.5
            - 67(n) - 1/3 = 0.33
            - 89(y) - 2/4 = 0.5
            - 34(n) - 2/5 = 0.4
            So Average Precision = (0+0.5+0.33+0.5+0.4 / 5) = 0.346

        - suppose after Reranking - 
            - 45(y) - precision (how many useful out of total docs fetched) =  1/1 = 1
            - 89(y) - 2/2 = 1
            - 23(n) - 2/3 = 0.66
            - 67(n) - 2/4 = 0.5
            - 34(n) - 2/5 = 0.4
            So Average Precision = (1+1+0.66+0.5+0.4 / 5) = 0.712


##### 2 - Pipelines :
- 1 : ***RAG Pipeline - Evaluating the entire process from retrieval to generation***

    - RAG Triad - 3 Evaluation matrics which evluates the entire Input query , Context and final Answer

    - As discussed above it contains three Evaluation matrices :
        - 1 : Faithfulness
        - 2 : Answer Relevancy
        - 3 : Contextual Relevancy

    - Only difference is in the Component level Evaluation we used the Context from the Golden Dataset and not the actual Context retrieved from the retriever


##### 3 - Application / System level : 